# Lab 2 — Turning the other knobs: sampling, stopping, and steering

**Artificial Intelligence · 4th year · UIR · Pr. Hakim Hafidi**

| | |
|---|---|
| **Duration** | 1 hour 30, individual work |
| **Prerequisite** | Lab 1 (you already have a `GEMINI_API_KEY` in Colab Secrets) |
| **You will learn** | what `top_p` and `top_k` actually change · how a *stop sequence* controls where the model halts · how `max_tokens` truncates and what that costs · how a `system` instruction steers behaviour |
| **Deliverable** | this notebook, executed, + the `lab02_<your id>.json` file it generates, both submitted on connect.uir.ac.ma |
| **Grading** | 4 points: it runs (1) · measurements are correct (1) · interpretation (1) · critical analysis (1) |

> In Lab 1 you turned **temperature** and swapped **models**. Those are two of the knobs.
> This lab turns the rest of them. Same toolbox, same `ask()` — with three new arguments.

> **Rule for the whole semester:** never paste an API key into a notebook cell.
> You already set up Colab Secrets in Lab 1 — nothing to redo there.

---
## Step 0 — Your API key (should already be done)

If you did Lab 1, your key is already stored. You do **not** need a new one.

1. Open the **🔑 key icon** in the left sidebar and confirm a secret named `GEMINI_API_KEY` exists, with **Notebook access** ON.
2. If it is missing: go to **https://aistudio.google.com/apikey**, create a key (no credit card), and add it as a secret named `GEMINI_API_KEY`.
3. Run the two cells below.

*If anything fails, raise your hand — do not spend ten minutes alone on it.*

In [ ]:
# Install the libraries this lab needs (~30 seconds)
%pip install -q google-genai pandas
print("done")

In [ ]:
#@title ⚙️ Toolbox — run this cell once, then fold it { display-mode: "form" }
# =============================================================================
#  AI course · UIR · shared lab toolbox  (v2)
#  Same as Lab 1, but ask() now also accepts: top_p, top_k, stop.
#  You do not need to modify this cell. Read the ask() docstring — the three
#  new arguments are what this lab is about.
# =============================================================================
import os, re, json, time, hashlib, pathlib, textwrap
from dataclasses import dataclass, asdict

# --- Models available on the Gemini free tier --------------------------------
# Names change every few months. If a call fails with "model not found",
# check https://ai.google.dev/gemini-api/docs/models and update these two lines.
MODEL_FAST   = "gemini-2.5-flash-lite"   # cheapest, highest daily quota
MODEL_STRONG = "gemini-2.5-flash"        # better, lower quota

# Public prices, USD per 1M tokens — used to *simulate* what your calls cost.
PRICES = {
    "gemini-2.5-flash-lite": (0.10, 0.40),
    "gemini-2.5-flash":      (0.30, 2.50),
}

CACHE_DIR = pathlib.Path("/content/.ai_cache"); CACHE_DIR.mkdir(exist_ok=True, parents=True)
CALL_LOG  = []          # every call made in this session
USE_LOCAL_FALLBACK = False   # set to True only if your API key does not work

@dataclass
class Answer:
    text: str; model: str; temperature: float
    in_tokens: int; out_tokens: int; latency_s: float; cost_usd: float
    top_p: float = None; top_k: int = None; finish: str = ""
    cached: bool = False
    def __str__(self): return self.text
    def __repr__(self): return self.text

def _api_key():
    """Read the key from Colab Secrets (preferred) or an env variable."""
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k: return k.strip()
    except Exception:
        pass
    return (os.environ.get("GEMINI_API_KEY") or "").strip()

_client = None
def _client_once():
    global _client
    if _client is None:
        from google import genai
        key = _api_key()
        if not key:
            raise RuntimeError(
                "No API key found. Add it in Colab: 🔑 (left sidebar) → + New secret →\n"
                "  Name: GEMINI_API_KEY   Value: <your key>   → enable 'Notebook access'."
            )
        _client = genai.Client(api_key=key)
    return _client

# --- Emergency fallback: a small open model running locally in Colab ---------
_local = None
def _local_once():
    global _local
    if _local is None:
        print("⚠️  Loading a small local model (slow, low quality — emergency use only)…")
        from transformers import pipeline
        _local = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
                          max_new_tokens=300, do_sample=True)
    return _local

def _cache_path(payload):
    return CACHE_DIR / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24] + ".json")

def ask(prompt, model=MODEL_FAST, temperature=0.0, max_tokens=800,
        system=None, top_p=None, top_k=None, stop=None,
        use_cache=True, retries=5, verbose=False):
    """Send one prompt to a model and return an Answer.

    Familiar from Lab 1:
      ask("Hello")                       -> fast model, temperature 0
      ask("Hello", model=MODEL_STRONG)   -> stronger model
      ask("Hello", temperature=1.5)      -> more random

    New in Lab 2:
      ask("Hello", top_p=0.5)            -> nucleus sampling: keep only the most
                                            probable tokens summing to 50% mass
      ask("Hello", top_k=5)              -> consider only the 5 likeliest tokens
      ask("Story.", stop=["\n"])         -> stop generating at the first newline
      ask("Hello", max_tokens=20)        -> cut the answer off after 20 tokens

    a.finish tells you WHY generation stopped: 'STOP' (natural end),
    'MAX_TOKENS' (hit the max_tokens ceiling), etc.

    Results are cached on disk: re-running a cell with the same prompt and the
    same settings is free and instant.
    """
    key = {"p": prompt, "m": model, "t": temperature, "mt": max_tokens,
           "s": system, "tp": top_p, "tk": top_k, "stop": stop}
    cp = _cache_path(key)
    if use_cache and cp.exists():
        d = json.loads(cp.read_text()); d["cached"] = True
        a = Answer(**d); CALL_LOG.append(asdict(a)); return a

    if USE_LOCAL_FALLBACK:
        t0 = time.time()
        out = _local_once()(prompt, temperature=max(temperature, 0.01))[0]["generated_text"]
        txt = out[len(prompt):].strip() if out.startswith(prompt) else out.strip()
        a = Answer(txt, "local-fallback", temperature, len(prompt)//4, len(txt)//4,
                   time.time()-t0, 0.0, top_p, top_k, "STOP")
        CALL_LOG.append(asdict(a)); return a

    from google.genai import types
    cfg = types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens,
                                      system_instruction=system, top_p=top_p, top_k=top_k,
                                      stop_sequences=stop)
    delay = 2.0
    for attempt in range(retries):
        try:
            t0 = time.time()
            r = _client_once().models.generate_content(model=model, contents=prompt, config=cfg)
            dt = time.time() - t0
            u  = getattr(r, "usage_metadata", None)
            ti = getattr(u, "prompt_token_count", 0) or 0
            to = getattr(u, "candidates_token_count", 0) or 0
            pi, po = PRICES.get(model, (0.0, 0.0))
            fin = ""
            try:
                fin = str(getattr(r.candidates[0], "finish_reason", "") or "").split(".")[-1]
            except Exception:
                pass
            a = Answer((r.text or "").strip(), model, temperature, ti, to, round(dt, 2),
                       round(ti/1e6*pi + to/1e6*po, 6), top_p, top_k, fin)
            cp.write_text(json.dumps({k: v for k, v in asdict(a).items() if k != "cached"}))
            CALL_LOG.append(asdict(a))
            if verbose: print(f"[{model}] {ti}→{to} tokens · {dt:.2f}s · {fin}")
            return a
        except Exception as e:
            msg = str(e)
            if any(s in msg for s in ("429", "RESOURCE_EXHAUSTED", "quota", "503", "UNAVAILABLE")):
                print(f"⏳ rate limit or server busy — waiting {delay:.0f}s "
                      f"(attempt {attempt+1}/{retries}). Free tier is ~15 calls/minute.")
                time.sleep(delay); delay *= 2; continue
            raise
    raise RuntimeError("Still rate-limited after several retries. Wait a minute, or use the cache.")

def usage_report():
    """Print what this session has cost you so far."""
    import pandas as pd
    if not CALL_LOG: print("No calls yet."); return None
    df = pd.DataFrame(CALL_LOG)
    real = df[~df.cached]
    print(f"calls: {len(df)}  (real: {len(real)}, from cache: {int(df.cached.sum())})")
    print(f"tokens in/out: {int(real.in_tokens.sum())} / {int(real.out_tokens.sum())}")
    print(f"time spent waiting: {real.latency_s.sum():.1f}s    simulated cost: ${real.cost_usd.sum():.5f}")
    return df

def check_environment():
    """Verify that everything needed for this lab is available."""
    ok = True
    try:
        import google.genai; print("✅ google-genai installed")
    except ImportError:
        print("❌ google-genai missing — run the install cell above"); ok = False
    if _api_key(): print("✅ API key found")
    else: print("❌ API key not found — see the instructions above"); ok = False
    try:
        a = ask("Reply with exactly: OK", max_tokens=10)
        print(f"✅ model answered: {a.text[:40]!r}")
    except Exception as e:
        print(f"❌ call failed: {str(e)[:200]}"); ok = False
    print("\n" + ("🎉 You are ready." if ok else "⚠️  Fix the ❌ above, then re-run this cell."))
    return ok

def export_lab(lab_id, student_id, answers: dict, tables: dict = None, extra: dict = None):
    """Build the file you submit on connect.uir.ac.ma."""
    import pandas as pd
    student_id = str(student_id).strip()
    assert student_id and student_id.lower() not in ("", "xxxxx", "your_id"), \
        "Set STUDENT_ID to your real apogee number."
    payload = {"lab": lab_id, "student_id": student_id,
               "answers": {k: str(v).strip() for k, v in answers.items()},
               "tables": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v)
                          for k, v in (tables or {}).items()},
               "extra": extra or {},
               "call_log": CALL_LOG,
               "created": time.strftime("%Y-%m-%d %H:%M")}
    name = f"{lab_id}_{student_id}.json"
    pathlib.Path(name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    empty = [k for k, v in payload["answers"].items() if len(v) < 40]
    print(f"📦 {name} written ({pathlib.Path(name).stat().st_size} bytes)")
    if empty: print(f"⚠️  These answers look too short: {empty}")
    else:     print("✅ All answers filled.")
    print("\nSubmit BOTH files on connect.uir.ac.ma:")
    print(f"   1. this notebook (File → Download → .ipynb)\n   2. {name}")
    return payload

print("✅ Toolbox v2 loaded. ask() now takes top_p, top_k, stop.")
print(f"   Models: MODEL_FAST={MODEL_FAST!r}  MODEL_STRONG={MODEL_STRONG!r}")

---
## Step 1 — Check that everything still works

Same check as Lab 1. If you see ✅ three times, you are ready.

In [ ]:
check_environment()

---
## Part A — `top_p` and `top_k`: the other randomness knobs (25 minutes)

Temperature is not the only thing that controls how random the model is. Before the model
picks the next token, it has a ranked list of candidates with probabilities. Two filters can
trim that list **before** temperature is even applied:

- **`top_k`** — keep only the *k* most likely tokens. `top_k=1` is greedy: always the single
  most likely token. `top_k=40` lets 40 candidates through.
- **`top_p`** (nucleus sampling) — keep the smallest set of top tokens whose probabilities
  add up to *p*. `top_p=1.0` keeps everything; `top_p=0.1` keeps only the near-certain ones.

The intuition: **temperature reshapes the dice, `top_k`/`top_p` decide how many faces the dice
even has.** Let's see the effect.

### A.1 — Does `top_p` change how varied the answers are?

We hold **temperature fixed at 1.0** (so there *is* randomness to filter) and change only
`top_p`. For each setting we make 4 real calls and count how many **distinct** answers come out.

> A low `top_p` should collapse the variety; a high `top_p` should let it bloom.

In [ ]:
QUESTION = "Suggest one catchy name for a student robotics club. Reply with only the name."

import pandas as pd
rowsA = []
for p in [0.1, 0.5, 0.95]:
    outs = []
    for i in range(4):
        # TODO (1 line): call ask() with QUESTION, temperature=1.0, top_p=p,
        #                use_cache=False, and append the .text to `outs`.
        outs.append(...)
    assert all(isinstance(t, str) for t in outs), "Fill the TODO: outs.append(ask(..., top_p=p, use_cache=False).text)"
    rowsA.append({"top_p": p, "distinct": len(set(outs)), "example": outs[0][:40]})

topp = pd.DataFrame(rowsA)
print(topp.to_string(index=False))

In [ ]:
assert len(topp) == 3, "You need one row per top_p value."
assert topp.distinct.max() >= 1, "Distinct counts are missing — did the calls run?"
print("✅ A.1 done — note the trend in the distinct column, you will need it in Part D (Q1).")

### A.2 — `top_k=1` is greedy

There is one setting that removes randomness entirely, no matter the temperature: `top_k=1`.
With only one candidate token allowed, the model has no choice to make. Test it: 3 calls at a
high temperature but `top_k=1`.

In [ ]:
greedy = []
for i in range(3):
    # TODO (1 line): ask() QUESTION at temperature=1.5, top_k=1, use_cache=False; append .text
    greedy.append(...)

assert all(isinstance(t, str) for t in greedy), "Fill the TODO line above."
print("distinct answers with top_k=1 at T=1.5 :", len(set(greedy)), "/3")
for g in greedy: print(" -", g[:60].replace(chr(10), ' '))

In [ ]:
assert len(greedy) == 3, "Need 3 runs."
print("✅ A.2 done.")

---
## Part B — Stop sequences: telling the model where to halt (20 minutes)

A **stop sequence** is a string that, the moment the model produces it, ends generation.
The stop string itself is **not** returned. This is how you get clean, bounded output without
post-processing — essential when the model feeds another program.

Two things to watch as you experiment:
- the **text** gets cut at the stop string, and
- `a.finish` reports **why** it stopped (`STOP` = natural end or a stop sequence hit,
  `MAX_TOKENS` = it ran into the length ceiling).

### B.1 — The same prompt, with and without a stop sequence

We ask for a numbered list. First with no stop, then with `stop=["2."]` so the model is cut
off the instant it starts item 2 — leaving you exactly the first item.

In [ ]:
LIST_PROMPT = "List three uses of AI in Moroccan agriculture, numbered 1., 2., 3."

full = ask(LIST_PROMPT, temperature=0.0, use_cache=False)

# TODO (1 line): ask() the SAME prompt and settings but with stop=["2."].
#                Store it in `stopped`.
stopped = ...

assert hasattr(stopped, "text"), "Fill the TODO line: stopped = ask(LIST_PROMPT, stop=[\"2.\"], use_cache=False)"
print("=== WITHOUT stop ==="); print(full.text)
print("\n=== WITH stop=['2.'] ==="); print(stopped.text)
print(f"\nout_tokens:  without={full.out_tokens}  with={stopped.out_tokens}")
print(f"finish:      without={full.finish!r}  with={stopped.finish!r}")

In [ ]:
assert '2.' not in stopped.text, "The stop sequence '2.' should not appear in the stopped answer."
assert stopped.out_tokens <= full.out_tokens, "The stopped answer should be no longer than the full one."
print("✅ B.1 done — the stopped answer is shorter and cheaper. Remember why for Q2.")

### B.2 — Why this matters: cost

You pay per **output** token. A stop sequence that trims useless trailing text is free money at
scale. Fill the tiny calculation below using your two measurements.

In [ ]:
# TODO (1 line): compute how many output tokens you SAVED per call by stopping early.
saved_tokens = ...

assert isinstance(saved_tokens, (int, float)), "saved_tokens = full.out_tokens - stopped.out_tokens"
print(f"tokens saved per call : {saved_tokens}")
print(f"over 100000 calls     : {saved_tokens * 100000:,} output tokens not paid for")

In [ ]:
assert saved_tokens >= 0, "saved_tokens should be full.out_tokens - stopped.out_tokens."
print("✅ B.2 done.")

---
## Part C — `max_tokens` and `system`: length and behaviour (25 minutes)

Two more knobs, both about **control** rather than randomness:

- **`max_tokens`** puts a hard ceiling on the answer length. Too low and the answer is cut
  mid-sentence — you can detect that from `a.finish == 'MAX_TOKENS'`.
- **`system`** is a separate instruction that sets the model's role and rules for the whole
  conversation, kept apart from the user's prompt.

### C.1 — Watch an answer get truncated

Ask for something that naturally wants a long answer, at three different `max_tokens` ceilings.
Record the finish reason each time.

In [ ]:
import pandas as pd
LONG_PROMPT = "Explain how a neural network learns, for a curious 15-year-old."

rowsC = []
for mt in [16, 64, 400]:
    # TODO (1 line): ask() LONG_PROMPT with max_tokens=mt, temperature=0.0, use_cache=False
    a = ...
    assert hasattr(a, "finish"), "Fill the TODO line: a = ask(LONG_PROMPT, max_tokens=mt, ...)"
    rowsC.append({"max_tokens": mt, "out_tokens": a.out_tokens,
                  "finish": a.finish, "ends_with": a.text[-40:].replace(chr(10), ' ')})

trunc = pd.DataFrame(rowsC)
print(trunc.to_string(index=False))

In [ ]:
assert len(trunc) == 3, "Need one row per max_tokens value."
assert trunc.out_tokens.min() > 0, "Token counts are zero — did the calls run?"
print("✅ C.1 done — look at which rows say MAX_TOKENS before answering Q3.")

### C.2 — A `system` instruction steers every answer

The `system` argument sets a standing rule. Send the **same** user prompt twice: once with no
system instruction, once with one that forces a fixed shape.

In [ ]:
USER_Q = "What is overfitting?"
SYSTEM_RULE = "You always answer in exactly one sentence, then stop."

plain = ask(USER_Q, temperature=0.0, use_cache=False)

# TODO (1 line): ask() the same USER_Q but pass system=SYSTEM_RULE. Store in `ruled`.
ruled = ...

assert hasattr(ruled, "text"), "Fill the TODO line: ruled = ask(USER_Q, system=SYSTEM_RULE, ...)"
print("=== NO system ==="); print(plain.text)
print(f"[{plain.out_tokens} output tokens]")
print("\n=== WITH system rule ==="); print(ruled.text)
print(f"[{ruled.out_tokens} output tokens]")

In [ ]:
assert plain.text and ruled.text, "Both answers must be non-empty."
print("✅ C.2 done — compare the two lengths and shapes for Q4.")

---
## Part D — Analysis (20 minutes)

Answer in the cell below, **in English, in your own words**, using your own numbers.
Two or three sentences each. This is where most of the grade is.

In [ ]:
STUDENT_ID = "XXXXX"   # TODO: your apogee number

# Q1 — In A.1 you varied top_p at fixed temperature. Describe the trend you saw in the
#      `distinct` column. In your own words, how is top_p different from temperature?
Q1 = """
...
"""

# Q2 — In Part B you added a stop sequence. What exactly changed in the output text and in
#      out_tokens? Give one concrete situation in your own field where a stop sequence would
#      be genuinely useful, and say which string you would stop on.
Q2 = """
...
"""

# Q3 — In C.1, which max_tokens values produced finish=='MAX_TOKENS'? Why is silently trusting
#      a truncated answer dangerous, and how would you detect truncation in a real application?
Q3 = """
...
"""

# Q4 — Compare the two answers in C.2 (with vs without the system rule), using your out_tokens.
#      Name one behaviour you would put in a system instruction rather than in each user prompt,
#      and explain why that is the better place for it.
Q4 = """
...
"""

In [ ]:
answers = {"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4}
problems = []
if STUDENT_ID.strip().upper() in ("XXXXX", ""): problems.append("STUDENT_ID is not filled in")
for k, v in answers.items():
    if len(v.strip()) < 120: problems.append(f"{k} is too short (write 2–3 sentences)")
    if "..." in v:           problems.append(f"{k} still contains the placeholder")
if problems:
    print("⚠️  Not ready to submit:"); [print("   -", p) for p in problems]
else:
    print("✅ Everything looks complete. Run the export cell below.")

---
## Submit

Run the cell below, then upload **both** the notebook and the `.json` file on connect.uir.ac.ma.

In [ ]:
usage_report()
print()
export_lab("lab02", STUDENT_ID,
           answers={"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4},
           tables={"top_p_variety": topp,
                   "truncation": trunc},
           extra={"greedy_distinct": len(set(greedy)),
                  "tokens_saved_by_stop": int(saved_tokens),
                  "plain_tokens": int(plain.out_tokens),
                  "ruled_tokens": int(ruled.out_tokens)})

---
## Going further (optional, not graded)

- Sweep `top_p` from 0.1 to 1.0 in steps of 0.1, 5 calls each, and plot distinct answers vs top_p.
  Where does variety saturate?
- Combine two stop sequences, e.g. `stop=["\n\n", "END"]`, and see which one fires first.
- Give a system instruction that fixes the **output language** ("Always answer in Arabic"), then
  send English prompts. Does the model obey across several very different questions?
- Re-run C.1 but read `a.finish` in a loop, raising `max_tokens` until it flips from
  `MAX_TOKENS` to `STOP`. That value is the natural length of the answer.